## 1. Manually created dataset

https://www.youtube.com/watch?v=N9hjO-Uy1Vo&list=PLfaIDFEXuae0um8Fj0V4dHG37fGFU8Q5S&index=3

**Code**: https://github.com/langchain-ai/langsmith-cookbook/blob/main/introduction/langsmith_introduction.ipynb

`Question:`

How can I build my own dataset?

`Setup:`

Let's build a dataset of question-answer pairs on this blog post about `DBRX`:

https://www.databricks.com/blog/introducing-dbrx-new-state-art-open-llm

We'll build a `Manually Curated` dataset of input, output pairs:

![image.png](attachment:image.png)

!pip install pandas

In [ ]:
from dotenv import load_dotenv

## os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "Test"
# os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
load_dotenv(dotenv_path='../.env')

In [ ]:
import pandas as pd

# QA
inputs = [
    "How many tokens was DBRX pre-trained on?",
    "Is DBRX a MOE model and how many parameters does it have?",
    "How many GPUs was DBRX trained on and what was the connectivity between GPUs?",
]

outputs = [
    "DBRX was pre-trained on 12 trillion tokens of text and code data.",
    "Yes, DBRX is a fine-grained mixture-of-experts (MoE) architecture with 132B total parameters.",
    "DBRX was trained on 3072 NVIDIA H100s connected by 3.2Tbps Infiniband",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "./data/DBRX_eval.csv"
df.to_csv(csv_path, index=False)

## Create new dataset and add data/example

- Use `dataset.id` to interact with dataset

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "DBRX"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="QA pairs about DBRX model.",
)
print(dataset)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

## Update dataset

- Use `dataset.id` to interact with

In [ ]:
new_questions = [
    "What is the context window of DBRX Instruct?",
]

new_answers = [
    "DBRX Instruct was trained with up to a 32K token context window.",
]

# See updated version in the UI
client.create_examples(
    inputs=[{"question": q} for q in new_questions],
    outputs=[{"answer": a} for a in new_answers],
    dataset_id=dataset.id,
)

# 2. Prepare DataSet from Traces

`Question:`

How can I save user logs as a dataset for future testing?

![image.png](attachment:image.png)

In [ ]:
# Load blog post

from dotenv import load_dotenv

## os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "Test"
# os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
load_dotenv(dotenv_path='../.env')

import requests
from bs4 import BeautifulSoup

url = "https://www.databricks.com/blog/introducing-dbrx-new-state-art-open-llm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")
text = [p.text for p in soup.find_all("p")]
full_text = "\n".join(text)

In [ ]:
len(full_text)

In [ ]:
# OpenAI API
import os
import openai
from langsmith.wrappers import wrap_openai

base_url = ""
api_key = os.environ['UNIFIED_LLM_KEY']

openai_client = wrap_openai(openai.Client(base_url=base_url, api_key=api_key))


def answer_dbrx_question_oai(inputs: dict) -> dict:
    """
    Generates answers to user questions based on a provided website text using OpenAI API.

    Parameters:
    inputs (dict): A dictionary with a single key 'question', representing the user's question as a string.

    Returns:
    dict: A dictionary with a single key 'output', containing the generated answer as a string.
    """

    # System prompt
    system_msg = (
        f"Answer user questions in 2-3 sentences about this context: \n\n\n {full_text}"
    )

    # Pass in website text
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": inputs["question"]},
    ]

    # Call OpenAI
    response = openai_client.chat.completions.create(
        messages=messages, model="gpt-4o"
    )

    # Response in output dict
    return {"answer": response.dict()["choices"][0]["message"]["content"]}

In [ ]:
# User question example

answer_dbrx_question_oai(
    {
        "question": "What are the main differences in training efficiency between MPT-7B vs DBRX?"
    }
)

In [ ]:
# User question example

answer_dbrx_question_oai({"question": "How many tokens was DBRX pre-trained on?"})

## Summary

- Select both records

- Create new dataset from it.

# 3. LLM-as-Judge: Built-in evaluator

`Question:`

How can I evaluate the my LLM against my dataset?

`Evaluation flow`

![image.png](attachment:image.png)

`Built-in evaluator`

https://docs.smith.langchain.com/evaluation/faq/evaluator-implementations

## Custom Evaluator Functions

In LangSmith, evaluators are **functions** that take a `Run` and `Example` and return a score dictionary.

### Evaluator Function Signature

In [ ]:
from langsmith.schemas import Run, Example

def my_evaluator(run: Run, example: Example) -> dict:
    """
    Custom evaluator function.

    Args:
        run: Contains outputs from the model
        example: Contains expected outputs

    Returns:
        Dictionary with 'key' and 'score'
    """
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")

    # Your evaluation logic here
    score = 1.0 if prediction == reference else 0.0

    return {
        "key": "metric_name",
        "score": score,
        "comment": "Optional explanation"  # Optional
    }

## Common Evaluator Patterns

| Evaluator Type | Description | Use Case |

|---|---|---|

| **Exact Match** | Strict equality check | When precision is critical |

| **Contains Answer** | Check if reference is in prediction | More lenient matching |

| **Word Overlap** | Semantic similarity via word overlap | Paraphrase detection |

| **Length-based** | Evaluate based on response length | Conciseness measurement |

| **Keyword Presence** | Check for important terms | Domain-specific validation |

| **LLM-as-Judge** | Use another LLM to evaluate | Complex quality assessment |

**💡 Quick Guide:**

- **Exact matching** → Use when correctness is binary

- **Lenient matching** → Use when paraphrases are acceptable

- **Custom logic** → Create evaluator functions for specific needs

- **Multiple metrics** → Pass multiple evaluators to get different perspectives

In [ ]:
from langsmith.evaluation import evaluate
from langsmith.schemas import Run, Example

# Create custom evaluator function
def qa_correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Evaluates if the answer matches the expected output.

    Args:
        run: Contains the model's output
        example: Contains the expected answer

    Returns:
        Dictionary with evaluation score
    """
    # Get prediction and reference
    prediction = run.outputs.get("answer", "").strip().lower()
    reference = example.outputs.get("answer", "").strip().lower()

    # Simple correctness check
    score = 1.0 if prediction == reference else 0.0

    return {
        "key": "correctness",
        "score": score,
        "comment": f"Predicted: '{prediction[:50]}...' vs Expected: '{reference[:50]}...'"
    }

# Run evaluation
dataset_name = 'DBRX'

experiment_results = evaluate(
    answer_dbrx_question_oai,
    data=dataset_name,
    evaluators=[qa_correctness_evaluator],  # Pass the function directly
    experiment_prefix="test_dbrx_question_oai",
    metadata={
        "variant": "stuff website context into gpt-4o"
    }
)

print("✓ Evaluation completed!")

## Using Different Evaluator Types

The evaluators are configured as dictionaries with `evaluator_type` key.

In [ ]:
# Different types of custom evaluator functions

from langsmith.schemas import Run, Example
from typing import Optional

# 1. Exact Match Evaluator
def exact_match_evaluator(run: Run, example: Example) -> dict:
    """Check if output exactly matches expected answer."""
    prediction = run.outputs.get("answer", "").strip()
    reference = example.outputs.get("answer", "").strip()

    score = 1.0 if prediction == reference else 0.0

    return {"key": "exact_match", "score": score}


# 2. Contains Answer Evaluator (more lenient)
def contains_answer_evaluator(run: Run, example: Example) -> dict:
    """Check if the output contains the expected answer."""
    prediction = run.outputs.get("answer", "").lower()
    reference = example.outputs.get("answer", "").lower()

    score = 1.0 if reference in prediction else 0.0

    return {"key": "contains_answer", "score": score}


# 3. Length-based Evaluator
def conciseness_evaluator(run: Run, example: Example) -> dict:
    """Evaluate based on answer length (conciseness)."""
    answer = run.outputs.get("answer", "")
    length = len(answer.split())

    # Score: 1.0 for 10-50 words, decreasing for longer answers
    if 10 <= length <= 50:
        score = 1.0
    elif length < 10:
        score = 0.5  # Too short
    else:
        score = max(0.0, 1.0 - (length - 50) / 100)  # Penalize long answers

    return {
        "key": "conciseness",
        "score": score,
        "comment": f"Answer length: {length} words"
    }


# 4. Semantic Similarity Evaluator (using simple word overlap)
def word_overlap_evaluator(run: Run, example: Example) -> dict:
    """Calculate word overlap between prediction and reference."""
    prediction_words = set(run.outputs.get("answer", "").lower().split())
    reference_words = set(example.outputs.get("answer", "").lower().split())

    if not reference_words:
        return {"key": "word_overlap", "score": 0.0}

    overlap = len(prediction_words & reference_words)
    score = overlap / len(reference_words)

    return {
        "key": "word_overlap",
        "score": score,
        "comment": f"Overlap: {overlap}/{len(reference_words)} words"
    }


# 5. Keyword Presence Evaluator
def keyword_evaluator(run: Run, example: Example) -> dict:
    """Check if important keywords are present."""
    answer = run.outputs.get("answer", "").lower()

    # Define important keywords (example)
    important_keywords = ["dbrx", "tokens", "parameters", "model"]

    present_keywords = sum(1 for kw in important_keywords if kw in answer)
    score = present_keywords / len(important_keywords)

    return {
        "key": "keyword_presence",
        "score": score,
        "comment": f"{present_keywords}/{len(important_keywords)} keywords found"
    }


print("✓ Multiple evaluator types defined")
print("\n💡 Usage:")
print("   evaluate(")
print("       target_function,")
print("       data=dataset,")
print("       evaluators=[exact_match_evaluator, contains_answer_evaluator],")
print("       ...")
print("   )")
print("\n📊 Available evaluators:")
print("   1. exact_match_evaluator - Strict matching")
print("   2. contains_answer_evaluator - Lenient matching")
print("   3. conciseness_evaluator - Based on length")
print("   4. word_overlap_evaluator - Semantic similarity")
print("   5. keyword_evaluator - Keyword presence")

# 4. Pairwise Evaluation

**Question:** How can I compare two different models or approaches side-by-side?

**Docs:** https://docs.langchain.com/langsmith/evaluate-pairwise

Pairwise evaluation allows you to compare two different systems (Model A vs Model B) on the same dataset and determine which performs better. This is useful for:

- **A/B Testing**: Compare two model versions

- **Model Selection**: Choose between different LLMs

- **Prompt Engineering**: Compare different prompts

- **Parameter Tuning**: Compare different temperature/hyperparameters

## How Pairwise Evaluation Works

Instead of evaluating a single output against a reference, pairwise evaluation:

1. Runs **both systems** on the same input

2. Compares the **two outputs** directly

3. Returns which one is better (or if they're equal)

In [ ]:
# Create two different model variants to compare

# Variant A: GPT-4o (original)
def answer_dbrx_question_v1(inputs: dict) -> dict:
    """Version 1: Using GPT-4o with original prompt."""
    system_msg = f"Answer user questions in 2-3 sentences about this context: \n\n\n {full_text}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": inputs["question"]},
    ]

    response = openai_client.chat.completions.create(
        messages=messages,
        model="gpt-4o",
        temperature=0
    )

    return {"answer": response.model_dump()["choices"][0]["message"]["content"]}


# Variant B: GPT-4o-mini (faster, cheaper)
def answer_dbrx_question_v2(inputs: dict) -> dict:
    """Version 2: Using GPT-4o-mini for faster/cheaper responses."""
    system_msg = f"Answer user questions in 2-3 sentences about this context: \n\n\n {full_text}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": inputs["question"]},
    ]

    response = openai_client.chat.completions.create(
        messages=messages,
        model="gpt-4o-mini",
        temperature=0
    )

    return {"answer": response.model_dump()["choices"][0]["message"]["content"]}

print("✓ Two model variants defined:")
print("  - Version 1: GPT-4o")
print("  - Version 2: GPT-4o-mini")

## ⚠️ IMPORTANT: Execute This Cell First

Before running the pairwise comparison below, make sure to:

1. **Execute the cell below** that defines `pairwise_preference_evaluator`

2. Then execute the cells to run Model A and Model B evaluations

3. Finally, execute the pairwise comparison cell

If you get a `ValueError` about the evaluator format, it means the kernel is using an old cached version of the function. **Restart the kernel** and re-run all cells in order.

In [ ]:
# Create a pairwise evaluator with correct format
from langsmith.schemas import Run, Example
from typing import Optional, Sequence

def pairwise_preference_evaluator(runs: Sequence[Run], example: Optional[Example] = None) -> dict:
    """
    Pairwise evaluator that compares two model outputs.

    Args:
        runs: Sequence of 2 Run objects (outputs from both models)
        example: The example with the question and expected answer

    Returns:
        Dictionary with 'key' and 'scores' mapping run IDs to scores
    """
    # Get outputs from both models
    output_a = runs[0].outputs.get("answer", "")
    output_b = runs[1].outputs.get("answer", "")

    # Get run IDs
    run_id_a = str(runs[0].id)
    run_id_b = str(runs[1].id)

    # Get reference if available
    reference = example.outputs.get("answer", "") if example else ""

    # Simple comparison: which one is closer to reference length?
    if reference:
        len_diff_a = abs(len(output_a.split()) - len(reference.split()))
        len_diff_b = abs(len(output_b.split()) - len(reference.split()))

        if len_diff_a < len_diff_b:
            # A is better
            score_a = 1.0
            score_b = 0.0
            reasoning = "Output A is closer to reference length"
        elif len_diff_b < len_diff_a:
            # B is better
            score_a = 0.0
            score_b = 1.0
            reasoning = "Output B is closer to reference length"
        else:
            # Tie
            score_a = 0.5
            score_b = 0.5
            reasoning = "Both outputs are equally close to reference length"
    else:
        # If no reference, compare lengths directly
        score_a = 0.5
        score_b = 0.5
        reasoning = "No reference available"

    # Return in the correct format for comparative evaluation
    return {
        "key": "pairwise_preference",
        "scores": {
            run_id_a: score_a,
            run_id_b: score_b
        },
        # Optional: add comment for debugging
        "comment": f"A: {len(output_a.split())} words, B: {len(output_b.split())} words, Ref: {len(reference.split()) if reference else 'N/A'} words - {reasoning}"
    }

print("✓ Pairwise evaluator created with correct format")
print("\n💡 Pairwise evaluator signature:")
print("   def evaluator(runs: Sequence[Run], example: Optional[Example]) -> dict")
print("\nReturns:")
print("   {")
print("       'key': 'metric_name',")
print("       'scores': {")
print("           'run_id_1': 1.0,  # First run score")
print("           'run_id_2': 0.0   # Second run score")
print("       }")
print("   }")

In [ ]:
# Step 1: Run evaluation for Model A (GPT-4o)
from langsmith.evaluation import evaluate

print("Running evaluation for Model A (GPT-4o)...")
results_a = evaluate(
    answer_dbrx_question_v1,
    data=dataset_name,
    evaluators=[],  # No evaluators needed for initial run
    experiment_prefix="model_a_gpt4o",
    metadata={"model": "gpt-4o", "variant": "A"}
)

print(f"✓ Model A evaluation complete")
print(f"  Experiment name: {results_a.experiment_name}")

# Step 2: Run evaluation for Model B (GPT-4o-mini)
print("\nRunning evaluation for Model B (GPT-4o-mini)...")
results_b = evaluate(
    answer_dbrx_question_v2,
    data=dataset_name,
    evaluators=[],  # No evaluators needed for initial run
    experiment_prefix="model_b_gpt4o_mini",
    metadata={"model": "gpt-4o-mini", "variant": "B"}
)

print(f"✓ Model B evaluation complete")
print(f"  Experiment name: {results_b.experiment_name}")

In [ ]:
# Step 3: Compare the two experiments using pairwise evaluation
from langsmith.evaluation import evaluate_comparative

print("\nRunning pairwise comparison...")
print(f"Comparing experiment '{results_a.experiment_name}' vs '{results_b.experiment_name}'")

# Compare using the experiment names (not IDs)
comparison_results = evaluate_comparative(
    (results_a.experiment_name, results_b.experiment_name),  # Positional: Tuple of experiment names
    evaluators=[pairwise_preference_evaluator],
    experiment_prefix="pairwise_comparison_gpt4o_vs_mini",
    metadata={
        "comparison": "GPT-4o vs GPT-4o-mini",
        "criteria": "length_similarity"
    }
)

print("✓ Pairwise comparison completed!")
print(f"\n📊 View comparison results in LangSmith UI")
print(f"   Comparison type: {type(comparison_results).__name__}")

## Advanced: LLM-as-Judge for Pairwise Evaluation

For more sophisticated comparisons, use an LLM to judge which output is better.

In [ ]:
# LLM-as-Judge pairwise evaluator with correct format
def llm_judge_pairwise(runs: Sequence[Run], example: Optional[Example] = None) -> dict:
    """
    Use an LLM to judge which of two outputs is better.

    This provides more nuanced evaluation than simple heuristics.
    """
    output_a = runs[0].outputs.get("answer", "")
    output_b = runs[1].outputs.get("answer", "")

    # Get run IDs
    run_id_a = str(runs[0].id)
    run_id_b = str(runs[1].id)

    # Get question and reference
    question = example.inputs.get("question", "") if example else ""
    reference = example.outputs.get("answer", "") if example else ""

    # Prompt for LLM judge
    judge_prompt = f"""You are an expert evaluator. Compare these two answers to the question and determine which is better.

Question: {question}

Reference Answer: {reference}

Answer A: {output_a}

Answer B: {output_b}

Evaluate based on:
1. Accuracy (does it match the reference?)
2. Completeness (does it fully answer the question?)
3. Clarity (is it well-written?)

Response format:
- If A is better, respond with: A|<reasoning>
- If B is better, respond with: B|<reasoning>
- If equal, respond with: TIE|<reasoning>

Your evaluation:"""

    # Call LLM judge
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0
    )

    judgment = response.model_dump()["choices"][0]["message"]["content"]

    # Parse response and assign scores
    if judgment.startswith("A|"):
        score_a = 1.0
        score_b = 0.0
        reasoning = judgment.split("|", 1)[1] if "|" in judgment else "A is better"
    elif judgment.startswith("B|"):
        score_a = 0.0
        score_b = 1.0
        reasoning = judgment.split("|", 1)[1] if "|" in judgment else "B is better"
    else:  # TIE
        score_a = 0.5
        score_b = 0.5
        reasoning = judgment.split("|", 1)[1] if "|" in judgment else "Equal quality"

    # Return in correct format for comparative evaluation
    return {
        "key": "llm_judge_pairwise",
        "scores": {
            run_id_a: score_a,
            run_id_b: score_b
        },
        "comment": f"Reasoning: {reasoning[:100]}..."
    }

print("✓ LLM-as-Judge pairwise evaluator created with correct format")
print("\n💡 This evaluator uses GPT-4o to compare outputs")
print("   Returns scores mapped to run IDs")
print("   Can evaluate multiple dimensions (accuracy, clarity, completeness)")

# Compare using the experiment names (not IDs)
comparison_results = evaluate_comparative(
    (results_a.experiment_name, results_b.experiment_name),  # Positional: Tuple of experiment names
    evaluators=[llm_judge_pairwise],
    experiment_prefix="pairwise_comparison_gpt4o_vs_mini",
    metadata={
        "comparison": "LLM as a judge - GPT-4o vs GPT-4o-mini",
        "criteria": "LLM as a judge"
    }
)

print("✓ Pairwise comparison completed!")
print(f"\n📊 View comparison results in LangSmith UI")
print(f"   Comparison type: {type(comparison_results).__name__}")

## Pairwise Evaluation Summary

| Comparison Method | When to Use | Pros | Cons |

|---|---|---|---|

| **Simple Heuristics** | Quick comparisons, clear metrics | Fast, cheap, deterministic | Limited nuance |

| **LLM-as-Judge** | Complex quality assessment | Nuanced, multi-dimensional | Slower, more expensive |

| **Statistical** | Large datasets, A/B tests | Objective, quantifiable | Requires large sample size |

| **Human Evaluation** | Final validation, edge cases | Most accurate | Expensive, slow |

**💡 Best Practices:**

1. **Start Simple**: Use heuristic evaluators first (length, keyword presence)

2. **Add LLM Judge**: For nuanced quality assessment

3. **Multiple Evaluators**: Combine different evaluation methods

4. **Analyze Results**: Look for patterns in which model wins

5. **Iterate**: Use insights to improve your models

**📊 In LangSmith UI:**

- View side-by-side comparison of outputs

- See win rates for each model

- Filter by evaluator scores

- Drill down into specific examples where models differ

## Human Voting in Pairwise Evaluation

**Question:** How can I collect human feedback on which output is better?

Human voting is the most accurate way to evaluate quality. You can collect human preferences and submit them to LangSmith for analysis.

In [ ]:
# Simple example: Collect human vote between two outputs
from langsmith import Client

client = Client()

# Example: You have two model outputs to compare
question = "How many tokens was DBRX pre-trained on?"
output_a = "DBRX was pre-trained on 12 trillion tokens of text and code data."
output_b = "DBRX was trained on 12T tokens."

print("Question:", question)
print("\nOutput A:", output_a)
print("Output B:", output_b)
print("\n" + "="*60)

# Simulate human vote (in real app, you'd collect this from users)
human_choice = "A"  # User prefers output A

print(f"Human vote: {human_choice} is better")
print("="*60)

# Convert to scores (1.0 for winner, 0.0 for loser)
if human_choice == "A":
    score_a = 1.0
    score_b = 0.0
elif human_choice == "B":
    score_a = 0.0
    score_b = 1.0
else:  # Tie
    score_a = 0.5
    score_b = 0.5

print(f"\n✓ Scores assigned: A={score_a}, B={score_b}")

# In production: Submit feedback to LangSmith
# You would need the actual run IDs from your experiments
print("\n💡 To submit to LangSmith:")
print("   1. Get run IDs from your evaluation runs")
print("   2. Use client.create_feedback() with the scores")
print("   3. Track human preferences over time")

### Complete Human Voting Workflow

Here's how to collect human votes and submit them to LangSmith:

In [ ]:
# Step 1: Run your evaluations and get run IDs
results_a = evaluate(model_a, data=dataset)
results_b = evaluate(model_b, data=dataset)

# Step 2: Get specific run IDs for comparison
# (From the experiment results or LangSmith UI)
run_id_a = "019c6152-fc6b-7ae3-8c00-15ea2b53ac41"
run_id_b = "019c6153-1e47-7b70-802d-212765c39c5f"

# Step 3: Collect human vote
print("Which output is better?")
print("A:", output_a)
print("B:", output_b)
user_vote = input("Enter A, B, or TIE: ").strip().upper()

# Step 4: Submit feedback to LangSmith
from langsmith import Client
client = Client()

if user_vote == "A":
    client.create_feedback(
        run_id=run_id_a,
        key="human_preference",
        score=1.0,
        comment="User preferred output A"
    )
    client.create_feedback(
        run_id=run_id_b,
        key="human_preference",
        score=0.0,
        comment="User preferred output A"
    )
elif user_vote == "B":
    client.create_feedback(
        run_id=run_id_a,
        key="human_preference",
        score=0.0,
        comment="User preferred output B"
    )
    client.create_feedback(
        run_id=run_id_b,
        key="human_preference",
        score=1.0,
        comment="User preferred output B"
    )
else:  # TIE
    client.create_feedback(
        run_id=run_id_a,
        key="human_preference",
        score=0.5,
        comment="Outputs are equal"
    )
    client.create_feedback(
        run_id=run_id_b,
        key="human_preference",
        score=0.5,
        comment="Outputs are equal"
    )

print("✓ Human feedback submitted to LangSmith!")

**💡 Use Cases:**

- **Quality Control**: Get human validation on model outputs

- **Model Selection**: Let humans choose between different models

- **Training Data**: Collect preference data for RLHF (Reinforcement Learning from Human Feedback)

- **A/B Testing**: Validate automated metrics with human judgment